In [ ]:
# Fahmida Kabir 
# Last Updated: 01/05/2025
# This specific file works on the facial recognition aspect for the robot canine, there are a lot of goals that the facial recognition will have to do, due to the time and speciality needed YOLO will be used.

In [ ]:
# Installing libraries required for the YOLO 5 to work 

#!pip install torch torchvision torchaudio
#!pip install opencv-python pandas
# pip install backports.tarfile


In [4]:
# Libraries that are needed
import os
import sys
import time
import math
import cv2
import numpy as np
import torch
import torch.nn as nn
import random
from collections import deque
import pickle
import threading
import queue
from pathlib import Path
# from gpiozero import Buzzer
# import smbus


In [5]:
# Setting up the buzzer
BUZZER_PIN = 17
GPIO.setmode(GPIO.BCM)
GPIO.setup(BUZZER_PIN, GPIO.OUT)

# Initialize Freenove Robot Dog
robot_dog = FreenoveRobotDog()

# Load YOLOv5 model
# 'yolov5n' is the smallest model

model = torch.hub.load('ultralytics/yolov5', 'yolov5n', device='cpu')
model.conf = 0.5

# OpenCV Video Capture on default camera [1]
cap = cv2.VideoCapture(0)

# Initialize face tracking
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')



NameError: name 'GPIO' is not defined

In [6]:
# Path Optimization class, using reinforcement learning with Q-learning

class PathOptimization:
    def __init__(self, actions=["forward", "backward", "left", "right"]):
        self.state = [0, 0]
        self.goal = [0, 0] 
        self.actions = actions
        # Empty dictionary to store Q values
        self.q_table = {}
        # Learning rate
        self.alpha = 0.1
        self.gamma = 0.9
        # Exploration rate
        self.epsilon = 0.1  

    def update_goal(self, ball_position):
        self.goal = ball_position
    
    def get_state(self):
        # Simple state representation based on position difference
        dx = self.goal[0] - self.state[0]
        dy = self.goal[1] - self.state[1]
        return (dx, dy)
    
    def choose_action(self):
        state = self.get_state()
        # If the state doesn't exist in the Q-table, initialize it with zeros
        if state not in self.q_table:
            self.q_table[state] = {action: 0 for action in self.actions}
        
        # Epsilon greedy Strategy
        # Chooses depending if its new to the enviroment or not between exploration and exploitation
        if random.uniform(0, 1) < self.epsilon:
            return random.choice(self.actions)
        else:
            q_values = self.q_table[state]
            return max(q_values, key=q_values.get)

    def move_towards_goal(self):
        dx = self.goal[0] - self.state[0]
        dy = self.goal[1] - self.state[1]
        action = self.choose_action()

        # Perform the action and update state accordingly, does so by using axis
        if action == "forward":
            robot_dog.move_forward()
            self.state[0] += np.sign(dx)
        elif action == "backward":
            robot_dog.move_backward()
            self.state[0] -= np.sign(dx)
        elif action == "left":
            robot_dog.turn_left()
            self.state[1] -= np.sign(dy)
        elif action == "right":
            robot_dog.turn_right()
            self.state[1] += np.sign(dy)

        # Calculate the reward: how close the robot is to the ball
        distance = np.linalg.norm([dx, dy])
        reward = -distance

        # Update Q-table using the Q-learning formula
        next_state = self.get_state()
        if next_state not in self.q_table:
            self.q_table[next_state] = {action: 0 for action in self.actions}
        
        best_next_action = max(self.q_table[next_state], key=self.q_table[next_state].get)
        td_target = reward + self.gamma * self.q_table[next_state][best_next_action]
        self.q_table[state][action] += self.alpha * (td_target - self.q_table[state][action])

        return self.state

# Initialize the path optimizer
path_optimizer = PathOptimization()


In [7]:
# Red ball detection function
def detect_red_ball(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    # The ball needs to be red to be found
    lower_red1 = np.array([0, 120, 70])
    upper_red1 = np.array([10, 255, 255])
    lower_red2 = np.array([170, 120, 70])
    upper_red2 = np.array([180, 255, 255])

    mask1 = cv2.inRange(hsv, lower_red1, upper_red1)
    mask2 = cv2.inRange(hsv, lower_red2, upper_red2)
    mask = mask1 | mask2

    # Finding the edges of the ball
    contours, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        c = max(contours, key=cv2.contourArea)
        if cv2.contourArea(c) > 500:
            x, y, w, h = cv2.boundingRect(c)
            return (x, y, w, h)
    return None

# Function to buzz when the ball is found
def buzz():
    for _ in range(3):
        GPIO.output(BUZZER_PIN, GPIO.HIGH)
        time.sleep(0.1)
        GPIO.output(BUZZER_PIN, GPIO.LOW)
        time.sleep(0.1)

# Face recognition function
def recognize_faces(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 4)
    face_labels = []
    
    # Will name faces a-z 
    for idx, (x, y, w, h) in enumerate(faces):
        label = f"Person {chr(65 + idx)}"
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cv2.putText(frame, label, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
        face_labels.append(label)
    
    return face_labels

In [ ]:
# Main code to bring everything together

# Using default camera [1]
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Object & face detection using YOLOv5
    results = model(frame)
    detections = results.pandas().xyxy[0]

    # Process detections
    for _, det in detections.iterrows():
        label = det['name']
        conf = det['confidence']
        x1, y1, x2, y2 = map(int, [det['xmin'], det['ymin'], det['xmax'], det['ymax']])
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, f"{label} {conf:.2f}", (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

        if label == 'person':
            # If a face is detected, label is placed
            recognize_faces(frame)

    # Red ball detection and tracking
    red_box = detect_red_ball(frame)
    if red_box:
        x, y, w, h = red_box
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 0, 255), 2)
        cv2.putText(frame, "Red Ball", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
        
        # Buzz when the ball is found
        buzz()

        # Update path optimizer with the ball position
        ball_position = [x + w // 2, y + h // 2]
        path_optimizer.update_goal(ball_position)
    
    # Making sure the end goal of the robot is to find the robot
    robot_position = path_optimizer.move_towards_goal()

    # Display the frame
    cv2.imshow("Robot Dog Vision", frame)

    # Stop record if q is pressed or robot is shut off
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
GPIO.cleanup()


NameError: name 'cap' is not defined

In [ ]:
# References 
# [1] GeeksforGeeks, “Python OpenCV: Capture Video from Camera,” GeeksforGeeks, Jan. 26, 2020. https://www.geeksforgeeks.org/python-opencv-capture-video-from-camera/ (accessed May 01, 2025).